# Build Core G10 FX Carry Panel

This notebook constructs the clean baseline G10 carry panel.

The baseline sample uses G10 currencies AUD, CAD, CHF, EUR, GBP, JPY, NOK, NZD, and SEK. DKK and HKD are left out of the first baseline because their peg or semi-peg behavior can make them less comparable to the free-floating G10 currencies.


## Why This Step Matters

FX carry returns are very sensitive to quote conventions. Some Bloomberg FX spot tickers are quoted as USD per foreign currency, such as AUDUSD or EURUSD. Others are quoted as foreign currency per USD, such as USDJPY or USDCAD. Before comparing carry across currencies, every quote must be converted into one common convention: **USD per 1 unit of foreign currency**.

Forward tickers also need diagnostics. A Bloomberg forward ticker can represent either an outright forward rate or forward points. Treating forward points as outright rates creates nonsensical carry signals and returns. This notebook therefore compares spot and forward levels by currency, classifies the 1M forward ticker as outright-like, points-like, missing, or ambiguous, and only then constructs the panel.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start] + list(start.parents):
        if (candidate / "src" / "bloomberg_data.py").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing src/bloomberg_data.py and data/.")


ROOT = find_project_root()
DATA_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "theo" / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Raw data folder: {DATA_DIR}")
print(f"Processed output folder: {PROCESSED_DIR}")


Project root: /Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo
Raw data folder: /Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/data/raw
Processed output folder: /Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/data/processed


## Configuration

The mapping below is explicit by design. `invert_to_usd_per_ccy=True` means the raw Bloomberg quote is foreign currency per USD, so the notebook uses its reciprocal to get USD per foreign currency.

The `point_size` column is used only when the forward ticker is classified as forward points. For JPY, one point is usually `0.01`; for the other G10 pairs here, one point is usually `0.0001` in the raw pair convention.


In [2]:
TENOR = "1M"
PRICE_FIELD = "PX_LAST"
SUPPORTED_TENORS = ["1W", "1M", "3M", "6M", "12M"]

G10_BASELINE = ["AUD", "CAD", "CHF", "EUR", "GBP", "JPY", "NOK", "NZD", "SEK"]
G10_OPTIONAL = ["DKK", "HKD"]

QUOTE_MAP = {
    "AUD": {"raw_pair": "AUDUSD", "raw_units": "USD per AUD", "invert_to_usd_per_ccy": False, "point_size": 0.0001},
    "CAD": {"raw_pair": "USDCAD", "raw_units": "CAD per USD", "invert_to_usd_per_ccy": True,  "point_size": 0.0001},
    "CHF": {"raw_pair": "USDCHF", "raw_units": "CHF per USD", "invert_to_usd_per_ccy": True,  "point_size": 0.0001},
    "EUR": {"raw_pair": "EURUSD", "raw_units": "USD per EUR", "invert_to_usd_per_ccy": False, "point_size": 0.0001},
    "GBP": {"raw_pair": "GBPUSD", "raw_units": "USD per GBP", "invert_to_usd_per_ccy": False, "point_size": 0.0001},
    "JPY": {"raw_pair": "USDJPY", "raw_units": "JPY per USD", "invert_to_usd_per_ccy": True,  "point_size": 0.01},
    "NOK": {"raw_pair": "USDNOK", "raw_units": "NOK per USD", "invert_to_usd_per_ccy": True,  "point_size": 0.0001},
    "NZD": {"raw_pair": "NZDUSD", "raw_units": "USD per NZD", "invert_to_usd_per_ccy": False, "point_size": 0.0001},
    "SEK": {"raw_pair": "USDSEK", "raw_units": "SEK per USD", "invert_to_usd_per_ccy": True,  "point_size": 0.0001},
    "DKK": {"raw_pair": "USDDKK", "raw_units": "DKK per USD", "invert_to_usd_per_ccy": True,  "point_size": 0.0001},
    "HKD": {"raw_pair": "USDHKD", "raw_units": "HKD per USD", "invert_to_usd_per_ccy": True,  "point_size": 0.0001},
}

# Optional manual override. Leave as "auto" for detection. Use only after manually verifying a ticker.
FORWARD_QUOTE_TYPE_OVERRIDE = {ccy: "auto" for ccy in G10_BASELINE + G10_OPTIONAL}

quote_map_table = pd.DataFrame([
    {
        "currency": ccy,
        "spot_ticker": f"{ccy} Curncy",
        "forward_ticker": f"{ccy}{TENOR} Curncy",
        **cfg,
        "manual_forward_type_override": FORWARD_QUOTE_TYPE_OVERRIDE.get(ccy, "auto"),
    }
    for ccy, cfg in QUOTE_MAP.items()
])

display(quote_map_table[quote_map_table["currency"].isin(G10_BASELINE)])


,currency,spot_ticker,forward_ticker,raw_pair,raw_units,invert_to_usd_per_ccy,point_size,manual_forward_type_override
0,AUD,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,0.0001,auto
1,CAD,CAD Curncy,CAD1M Curncy,USDCAD,CAD per USD,True,0.0001,auto
2,CHF,CHF Curncy,CHF1M Curncy,USDCHF,CHF per USD,True,0.0001,auto
3,EUR,EUR Curncy,EUR1M Curncy,EURUSD,USD per EUR,False,0.0001,auto
4,GBP,GBP Curncy,GBP1M Curncy,GBPUSD,USD per GBP,False,0.0001,auto
5,JPY,JPY Curncy,JPY1M Curncy,USDJPY,JPY per USD,True,0.0100,auto
6,NOK,NOK Curncy,NOK1M Curncy,USDNOK,NOK per USD,True,0.0001,auto
7,NZD,NZD Curncy,NZD1M Curncy,NZDUSD,USD per NZD,False,0.0001,auto
8,SEK,SEK Curncy,SEK1M Curncy,USDSEK,SEK per USD,True,0.0001,auto


## Load Saved Bloomberg Data

This notebook reads the saved long-format `g10_fx_spot_forward` file only. It does not call Bloomberg. If the parquet file is unavailable or the environment lacks a parquet engine, the loader tries a CSV mirror and reports a warning instead of failing silently.


In [3]:
def empty_long_frame():
    return pd.DataFrame(columns=["date", "ticker", "field", "value"])


def read_long_group(group, data_dir=DATA_DIR):
    parquet_path = data_dir / f"{group}_long.parquet"
    csv_path = data_dir / f"{group}_long.csv"
    attempts = []
    for path, reader in [(parquet_path, pd.read_parquet), (csv_path, pd.read_csv)]:
        if not path.exists():
            attempts.append(f"missing {path.name}")
            continue
        try:
            df = reader(path)
            needed = ["date", "ticker", "field", "value"]
            missing_cols = [c for c in needed if c not in df.columns]
            if missing_cols:
                raise ValueError(f"File is missing required columns: {missing_cols}")
            df = df[needed].copy()
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df["ticker"] = df["ticker"].astype("string")
            df["field"] = df["field"].astype("string")
            df["value"] = pd.to_numeric(df["value"], errors="coerce")
            return df, {"loaded": True, "path": str(path.relative_to(ROOT)), "rows": len(df), "message": ""}
        except Exception as exc:
            attempts.append(f"{path.name}: {repr(exc)}")
    return empty_long_frame(), {"loaded": False, "path": "", "rows": 0, "message": "; ".join(attempts)}


g10_long, load_status = read_long_group("g10_fx_spot_forward")
display(pd.DataFrame([load_status]))

if g10_long.empty:
    warnings.warn("No G10 spot/forward data loaded. Later cells will create empty diagnostic tables rather than crash.")
else:
    display(g10_long.head())
    print(f"Rows: {len(g10_long):,}")
    print(f"Tickers: {g10_long['ticker'].nunique():,}")
    print(f"Fields: {sorted(g10_long['field'].dropna().unique())}")


,loaded,path,rows,message
0,True,data/raw/g10_fx_spot_forward_long.parquet,1007085,


,date,ticker,field,value
0,2007-01-01,AUD Curncy,PX_LAST,0.7894
1,2007-01-01,AUD Curncy,PX_BID,0.7893
2,2007-01-01,AUD Curncy,PX_ASK,0.7895
3,2007-01-02,AUD Curncy,PX_LAST,0.7961
4,2007-01-02,AUD Curncy,PX_BID,0.7960


Rows: 1,007,085
Tickers: 66
Fields: ['PX_ASK', 'PX_BID', 'PX_LAST']


## Ticker Availability

The baseline requires each currency's spot ticker and selected-tenor forward ticker. Bid and ask fields are not used for this first clean `PX_LAST` panel, but their availability is checked because they matter for later transaction-cost adjustments.


In [4]:
def ticker_field_availability(df, currencies, tenor=TENOR):
    rows = []
    available_pairs = set(zip(df["ticker"].astype(str), df["field"].astype(str))) if not df.empty else set()
    available_tickers = set(df["ticker"].dropna().astype(str).unique()) if not df.empty else set()
    for ccy in currencies:
        spot_ticker = f"{ccy} Curncy"
        fwd_ticker = f"{ccy}{tenor} Curncy"
        rows.append({
            "currency": ccy,
            "spot_ticker": spot_ticker,
            "forward_ticker": fwd_ticker,
            "spot_ticker_available": spot_ticker in available_tickers,
            "forward_ticker_available": fwd_ticker in available_tickers,
            "spot_PX_LAST": (spot_ticker, "PX_LAST") in available_pairs,
            "forward_PX_LAST": (fwd_ticker, "PX_LAST") in available_pairs,
            "spot_bid_ask": (spot_ticker, "PX_BID") in available_pairs and (spot_ticker, "PX_ASK") in available_pairs,
            "forward_bid_ask": (fwd_ticker, "PX_BID") in available_pairs and (fwd_ticker, "PX_ASK") in available_pairs,
        })
    return pd.DataFrame(rows)

availability = ticker_field_availability(g10_long, G10_BASELINE, TENOR)
display(availability)

missing_required = availability[~(availability["spot_PX_LAST"] & availability["forward_PX_LAST"])]
if not missing_required.empty:
    warnings.warn("Some baseline currencies are missing required PX_LAST spot or forward data. They will be excluded from the constructed panel.")
    display(missing_required)


,currency,spot_ticker,forward_ticker,spot_ticker_available,forward_ticker_available,spot_PX_LAST,forward_PX_LAST,spot_bid_ask,forward_bid_ask
0,AUD,AUD Curncy,AUD1M Curncy,True,True,True,True,True,True
1,CAD,CAD Curncy,CAD1M Curncy,True,True,True,True,True,True
2,CHF,CHF Curncy,CHF1M Curncy,True,True,True,True,True,True
3,EUR,EUR Curncy,EUR1M Curncy,True,True,True,True,True,True
4,GBP,GBP Curncy,GBP1M Curncy,True,True,True,True,True,True
5,JPY,JPY Curncy,JPY1M Curncy,True,True,True,True,True,True
6,NOK,NOK Curncy,NOK1M Curncy,True,True,True,True,True,True
7,NZD,NZD Curncy,NZD1M Curncy,True,True,True,True,True,True
8,SEK,SEK Curncy,SEK1M Curncy,True,True,True,True,True,True


## Forward Outright vs Forward Points Diagnostic

The diagnostic below compares spot and forward levels in the raw Bloomberg quote convention. For a 1M outright forward, the forward level should be very close to the spot level. Forward points can be small, negative, or much smaller/larger than the spot level depending on pair conventions. Ambiguous tickers are left visible for manual review.


In [5]:
def get_price_series(df, ticker, field=PRICE_FIELD):
    if df.empty:
        return pd.Series(dtype="float64", name=ticker)
    out = (df[(df["ticker"].astype(str) == ticker) & (df["field"].astype(str) == field)]
           .dropna(subset=["date"])
           .sort_values("date")
           .drop_duplicates(subset=["date"], keep="last")
           .set_index("date")["value"])
    out.name = ticker
    return pd.to_numeric(out, errors="coerce")


def classify_forward_quote(spot, forward):
    joined = pd.concat([spot.rename("spot"), forward.rename("forward_input")], axis=1).dropna()
    if joined.empty or len(joined) < 24:
        return "missing_or_too_short", joined
    rel_diff = ((joined["forward_input"] - joined["spot"]) / joined["spot"]).replace([np.inf, -np.inf], np.nan).dropna()
    ratio = (joined["forward_input"] / joined["spot"]).replace([np.inf, -np.inf], np.nan).dropna()
    negative_share = float((joined["forward_input"] < 0).mean())
    median_abs_rel_diff = float(rel_diff.abs().median()) if len(rel_diff) else np.nan
    median_ratio = float(ratio.median()) if len(ratio) else np.nan

    if pd.notna(median_abs_rel_diff) and negative_share == 0 and 0.98 <= median_ratio <= 1.02 and median_abs_rel_diff <= 0.02:
        return "outright_like", joined
    if negative_share > 0 or (pd.notna(median_abs_rel_diff) and median_abs_rel_diff > 0.02):
        return "points_like", joined
    return "ambiguous", joined


def build_forward_diagnostic(currencies, tenor=TENOR):
    rows = []
    pair_data = {}
    for ccy in currencies:
        spot_ticker = f"{ccy} Curncy"
        fwd_ticker = f"{ccy}{tenor} Curncy"
        spot = get_price_series(g10_long, spot_ticker)
        fwd = get_price_series(g10_long, fwd_ticker)
        classification, joined = classify_forward_quote(spot, fwd)
        pair_data[ccy] = joined
        if joined.empty:
            rows.append({
                "currency": ccy, "spot_ticker": spot_ticker, "forward_ticker": fwd_ticker,
                "observations": 0, "classification": classification,
                "median_spot": np.nan, "median_forward_input": np.nan,
                "median_forward_over_spot": np.nan, "median_abs_relative_difference": np.nan,
                "negative_forward_share": np.nan, "point_size_if_points": QUOTE_MAP[ccy]["point_size"],
            })
            continue
        rel_diff = ((joined["forward_input"] - joined["spot"]) / joined["spot"]).replace([np.inf, -np.inf], np.nan)
        ratio = (joined["forward_input"] / joined["spot"]).replace([np.inf, -np.inf], np.nan)
        rows.append({
            "currency": ccy,
            "spot_ticker": spot_ticker,
            "forward_ticker": fwd_ticker,
            "observations": len(joined),
            "classification": classification,
            "median_spot": joined["spot"].median(),
            "median_forward_input": joined["forward_input"].median(),
            "median_forward_over_spot": ratio.median(),
            "median_abs_relative_difference": rel_diff.abs().median(),
            "negative_forward_share": (joined["forward_input"] < 0).mean(),
            "point_size_if_points": QUOTE_MAP[ccy]["point_size"],
        })
    return pd.DataFrame(rows), pair_data


forward_diagnostic, raw_pair_data = build_forward_diagnostic(G10_BASELINE, TENOR)
display(forward_diagnostic)

ambiguous = forward_diagnostic[forward_diagnostic["classification"].isin(["ambiguous", "missing_or_too_short"])]
if not ambiguous.empty:
    warnings.warn("At least one forward ticker is ambiguous or too short. It will not be used unless a manual override is provided.")
    display(ambiguous)


,currency,spot_ticker,forward_ticker,observations,classification,median_spot,median_forward_input,median_forward_over_spot,median_abs_relative_difference,negative_forward_share,point_size_if_points
0,AUD,AUD Curncy,AUD1M Curncy,5087,points_like,0.75790,-5.98,-7.826547,9.903718,0.599174,0.0001
1,CAD,CAD Curncy,CAD1M Curncy,5087,points_like,1.26670,-0.71,-0.566414,5.321839,0.560645,0.0001
2,CHF,CHF Curncy,CHF1M Curncy,5087,points_like,0.95640,-12.30,-12.672121,13.672121,0.999803,0.0001
3,EUR,EUR Curncy,EUR1M Curncy,5087,points_like,1.17840,8.59,7.301702,9.413067,0.190486,0.0001
4,GBP,GBP Curncy,GBP1M Curncy,5087,points_like,1.39620,-0.38,-0.259879,3.163007,0.523884,0.0001
5,JPY,JPY Curncy,JPY1M Curncy,5087,points_like,109.59000,-11.25,-0.102372,1.102372,1.000000,0.0100
6,NOK,NOK Curncy,NOK1M Curncy,5082,points_like,8.18525,10.54,1.160210,9.271427,0.446084,0.0001
7,NZD,NZD Curncy,NZD1M Curncy,5087,points_like,0.69640,-9.45,-13.023578,14.095238,0.733635,0.0001
8,SEK,SEK Curncy,SEK1M Curncy,5087,points_like,8.41620,-55.75,-6.573454,13.213050,0.705131,0.0001


## Convert Raw Quotes to USD per Foreign Currency

For each currency, the notebook first constructs the forward outright in the raw Bloomberg pair convention. If the forward input is classified as points, it adds `forward_points * point_size` to the raw spot level. Then both spot and forward are converted to USD per foreign currency using the explicit quote-direction mapping.


In [6]:
def to_usd_per_ccy(raw_quote, invert):
    raw_quote = pd.to_numeric(raw_quote, errors="coerce")
    if invert:
        return 1.0 / raw_quote
    return raw_quote


def resolve_forward_type(ccy, detected_type):
    override = FORWARD_QUOTE_TYPE_OVERRIDE.get(ccy, "auto")
    if override == "auto":
        return detected_type
    if override not in {"points_like", "outright_like"}:
        warnings.warn(f"Ignoring invalid forward type override for {ccy}: {override}")
        return detected_type
    return override


def build_daily_currency_panel(currencies, tenor=TENOR):
    pieces = []
    construction_rows = []
    diag_lookup = forward_diagnostic.set_index("currency")["classification"].to_dict() if not forward_diagnostic.empty else {}

    for ccy in currencies:
        cfg = QUOTE_MAP[ccy]
        raw = raw_pair_data.get(ccy, pd.DataFrame()).copy()
        detected_type = diag_lookup.get(ccy, "missing_or_too_short")
        forward_type = resolve_forward_type(ccy, detected_type)

        if raw.empty or forward_type in {"ambiguous", "missing_or_too_short"}:
            construction_rows.append({
                "currency": ccy,
                "used": False,
                "detected_forward_type": detected_type,
                "forward_type_used": forward_type,
                "reason": "missing/ambiguous spot-forward pair",
            })
            continue

        if forward_type == "points_like":
            raw["raw_forward_outright"] = raw["spot"] + raw["forward_input"] * cfg["point_size"]
        elif forward_type == "outright_like":
            raw["raw_forward_outright"] = raw["forward_input"]
        else:
            construction_rows.append({
                "currency": ccy,
                "used": False,
                "detected_forward_type": detected_type,
                "forward_type_used": forward_type,
                "reason": "unsupported forward type",
            })
            continue

        raw["spot_usd_per_ccy"] = to_usd_per_ccy(raw["spot"], cfg["invert_to_usd_per_ccy"])
        raw["forward_1m_usd_per_ccy"] = to_usd_per_ccy(raw["raw_forward_outright"], cfg["invert_to_usd_per_ccy"])
        raw = raw.replace([np.inf, -np.inf], np.nan)

        valid = raw[(raw["spot_usd_per_ccy"] > 0) & (raw["forward_1m_usd_per_ccy"] > 0)].copy()
        dropped = len(raw) - len(valid)
        if valid.empty:
            construction_rows.append({
                "currency": ccy,
                "used": False,
                "detected_forward_type": detected_type,
                "forward_type_used": forward_type,
                "reason": "no positive converted spot/forward observations",
            })
            continue

        valid = valid.reset_index().rename(columns={"date": "date", "spot": "raw_spot", "forward_input": "raw_forward_input"})
        valid["currency"] = ccy
        valid["tenor"] = tenor
        valid["spot_ticker"] = f"{ccy} Curncy"
        valid["forward_ticker"] = f"{ccy}{tenor} Curncy"
        valid["raw_pair"] = cfg["raw_pair"]
        valid["raw_units"] = cfg["raw_units"]
        valid["invert_to_usd_per_ccy"] = cfg["invert_to_usd_per_ccy"]
        valid["forward_quote_type"] = forward_type
        valid["point_size_used"] = cfg["point_size"] if forward_type == "points_like" else np.nan

        pieces.append(valid[[
            "date", "currency", "tenor", "spot_ticker", "forward_ticker", "raw_pair", "raw_units",
            "invert_to_usd_per_ccy", "forward_quote_type", "point_size_used", "raw_spot",
            "raw_forward_input", "raw_forward_outright", "spot_usd_per_ccy", "forward_1m_usd_per_ccy",
        ]])
        construction_rows.append({
            "currency": ccy,
            "used": True,
            "detected_forward_type": detected_type,
            "forward_type_used": forward_type,
            "reason": "",
            "daily_observations": len(valid),
            "dropped_nonpositive_or_invalid": dropped,
        })

    if pieces:
        daily_panel = pd.concat(pieces, ignore_index=True).sort_values(["currency", "date"])
    else:
        daily_panel = pd.DataFrame(columns=[
            "date", "currency", "tenor", "spot_ticker", "forward_ticker", "raw_pair", "raw_units",
            "invert_to_usd_per_ccy", "forward_quote_type", "point_size_used", "raw_spot",
            "raw_forward_input", "raw_forward_outright", "spot_usd_per_ccy", "forward_1m_usd_per_ccy",
        ])
    return daily_panel, pd.DataFrame(construction_rows)


daily_panel, construction_report = build_daily_currency_panel(G10_BASELINE, TENOR)
display(construction_report)
display(daily_panel.head())


,currency,used,detected_forward_type,forward_type_used,reason,daily_observations,dropped_nonpositive_or_invalid
0,AUD,True,points_like,points_like,,5087,0
1,CAD,True,points_like,points_like,,5087,0
2,CHF,True,points_like,points_like,,5087,0
3,EUR,True,points_like,points_like,,5087,0
4,GBP,True,points_like,points_like,,5087,0
5,JPY,True,points_like,points_like,,5087,0
6,NOK,True,points_like,points_like,,5082,0
7,NZD,True,points_like,points_like,,5087,0
8,SEK,True,points_like,points_like,,5087,0


,date,currency,tenor,spot_ticker,forward_ticker,raw_pair,raw_units,invert_to_usd_per_ccy,forward_quote_type,point_size_used,raw_spot,raw_forward_input,raw_forward_outright,spot_usd_per_ccy,forward_1m_usd_per_ccy
0,2007-01-01,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.7894,-6.57,0.788743,0.7894,0.788743
1,2007-01-02,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.7961,-6.45,0.795455,0.7961,0.795455
2,2007-01-03,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.7912,-6.38,0.790562,0.7912,0.790562
3,2007-01-04,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.7844,-6.32,0.783768,0.7844,0.783768
4,2007-01-05,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.7787,-6.30,0.778070,0.7787,0.778070


## Quote-Direction Sanity Checks

After conversion, all `spot_usd_per_ccy` values should be positive and economically plausible. For example, JPY should be around cents per yen, while EUR and GBP should be above one USD per unit for much of the sample.


In [7]:
PLAUSIBLE_USD_PER_CCY_RANGES = {
    "AUD": (0.30, 1.50),
    "CAD": (0.50, 1.20),
    "CHF": (0.50, 1.60),
    "EUR": (0.60, 2.00),
    "GBP": (0.80, 2.50),
    "JPY": (0.004, 0.020),
    "NOK": (0.05, 0.25),
    "NZD": (0.30, 1.20),
    "SEK": (0.05, 0.25),
}

if daily_panel.empty:
    quote_sanity = pd.DataFrame(columns=["currency", "min_spot_usd_per_ccy", "median_spot_usd_per_ccy", "max_spot_usd_per_ccy", "latest_spot_usd_per_ccy", "plausible_range", "passes_range_check"])
else:
    rows = []
    for ccy, sub in daily_panel.groupby("currency"):
        lo, hi = PLAUSIBLE_USD_PER_CCY_RANGES.get(ccy, (0, np.inf))
        latest = sub.sort_values("date").iloc[-1]
        rows.append({
            "currency": ccy,
            "min_spot_usd_per_ccy": sub["spot_usd_per_ccy"].min(),
            "median_spot_usd_per_ccy": sub["spot_usd_per_ccy"].median(),
            "max_spot_usd_per_ccy": sub["spot_usd_per_ccy"].max(),
            "latest_date": latest["date"].date().isoformat(),
            "latest_spot_usd_per_ccy": latest["spot_usd_per_ccy"],
            "plausible_range": f"[{lo}, {hi}]",
            "passes_range_check": bool((sub["spot_usd_per_ccy"].dropna().between(lo, hi)).all()),
        })
    quote_sanity = pd.DataFrame(rows)

display(quote_sanity)

failed_range = quote_sanity[~quote_sanity["passes_range_check"]] if not quote_sanity.empty else quote_sanity
if not failed_range.empty:
    warnings.warn("Some converted spot values fall outside the broad sanity ranges. Review quote direction before using those currencies.")
    display(failed_range)


,currency,min_spot_usd_per_ccy,median_spot_usd_per_ccy,max_spot_usd_per_ccy,latest_date,latest_spot_usd_per_ccy,plausible_range,passes_range_check
0,AUD,0.574300,0.757900,1.102000,2026-06-30,0.691900,"[0.3, 1.5]",True
1,CAD,0.685918,0.789453,1.086484,2026-06-30,0.704424,"[0.5, 1.2]",True
2,CHF,0.797766,1.045588,1.387155,2026-06-30,1.237011,"[0.5, 1.6]",True
3,EUR,0.959400,1.178400,1.599100,2026-06-30,1.142200,"[0.6, 2.0]",True
4,GBP,1.068900,1.396200,2.107500,2026-06-30,1.326200,"[0.8, 2.5]",True
5,JPY,0.006152,0.009125,0.013189,2026-06-30,0.006152,"[0.004, 0.02]",True
6,NOK,0.085433,0.122171,0.201459,2026-06-30,0.100986,"[0.05, 0.25]",True
7,NZD,0.492700,0.696400,0.882300,2026-06-30,0.567800,"[0.3, 1.2]",True
8,SEK,0.087986,0.118818,0.171218,2026-06-30,0.103077,"[0.05, 0.25]",True


## Build Monthly 1M Carry Panel
The panel uses the last available observation in each calendar month. The carry signal is the return that would be earned over the month if the future spot rate equaled today's spot rate:$$\text{carry}_{t}^{1m} = \frac{S_t}{F_t^{1m}} - 1$$where both $S_t$ and $F_t^{1m}$ are expressed as USD per foreign currency. The realized one-month forward excess return is:$$\text{rx}_{t+1}^{1m} = \frac{S_{t+1}}{F_t^{1m}} - 1$$This is the payoff from buying the foreign currency forward at month-end $t$ and settling against the next month-end spot observation. It is a clean monthly approximation for research-panel construction.


In [8]:
def build_monthly_panel(daily):
    if daily.empty:
        return pd.DataFrame(columns=[
            "month_end", "currency", "tenor", "spot_ticker", "forward_ticker", "raw_pair", "raw_units",
            "invert_to_usd_per_ccy", "forward_quote_type", "point_size_used", "raw_spot",
            "raw_forward_input", "raw_forward_outright", "spot_usd_per_ccy", "forward_1m_usd_per_ccy",
            "carry_signal_1m", "next_month_spot_usd_per_ccy", "realized_1m_forward_excess_return",
        ])

    monthly_pieces = []
    for ccy, sub in daily.sort_values("date").groupby("currency"):
        sub_monthly = sub.set_index("date").resample("ME").last()
        sub_monthly["currency"] = ccy
        monthly_pieces.append(sub_monthly.reset_index().rename(columns={"date": "month_end"}))

    monthly = pd.concat(monthly_pieces, ignore_index=True) if monthly_pieces else pd.DataFrame()
    if monthly.empty:
        return build_monthly_panel(pd.DataFrame())

    monthly = monthly.sort_values(["currency", "month_end"])
    monthly["carry_signal_1m"] = monthly["spot_usd_per_ccy"] / monthly["forward_1m_usd_per_ccy"] - 1.0
    monthly["next_month_spot_usd_per_ccy"] = monthly.groupby("currency")["spot_usd_per_ccy"].shift(-1)
    monthly["realized_1m_forward_excess_return"] = monthly["next_month_spot_usd_per_ccy"] / monthly["forward_1m_usd_per_ccy"] - 1.0
    return monthly


monthly_panel = build_monthly_panel(daily_panel)
print(f"Monthly panel rows: {len(monthly_panel):,}")
display(monthly_panel.head(12))


Monthly panel rows: 2,106


,month_end,currency,tenor,spot_ticker,forward_ticker,raw_pair,raw_units,invert_to_usd_per_ccy,forward_quote_type,point_size_used,raw_spot,raw_forward_input,raw_forward_outright,spot_usd_per_ccy,forward_1m_usd_per_ccy,carry_signal_1m,next_month_spot_usd_per_ccy,realized_1m_forward_excess_return
0,2007-01-31,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.7769,-5.63,0.776337,0.7769,0.776337,0.000725,0.7879,0.014894
1,2007-02-28,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.7879,-6.35,0.787265,0.7879,0.787265,0.000807,0.8086,0.027100
2,2007-03-31,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.8086,-6.98,0.807902,0.8086,0.807902,0.000864,0.8301,0.027476
3,2007-04-30,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.8301,-6.95,0.829405,0.8301,0.829405,0.000838,0.8279,-0.001815
4,2007-05-31,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.8279,-6.49,0.827251,0.8279,0.827251,0.000785,0.8494,0.026774
5,2007-06-30,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.8494,-6.78,0.848722,0.8494,0.848722,0.000799,0.8517,0.003509
6,2007-07-31,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.8517,-8.39,0.850861,0.8517,0.850861,0.000986,0.8186,-0.037916
7,2007-08-31,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.8186,-6.85,0.817915,0.8186,0.817915,0.000837,0.8879,0.085565
8,2007-09-30,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.8879,-11.83,0.886717,0.8879,0.886717,0.001334,0.9341,0.053436
9,2007-10-31,AUD,1M,AUD Curncy,AUD1M Curncy,AUDUSD,USD per AUD,False,points_like,0.0001,0.9341,-15.65,0.932535,0.9341,0.932535,0.001678,0.8844,-0.051617


## Monthly Ranking Checks

These rankings are only sanity checks. They help confirm that the carry signal varies sensibly across currencies and over time. They are not a final trading strategy.


In [9]:
def top_bottom_by_month(panel, n=3):
    if panel.empty or "carry_signal_1m" not in panel:
        return pd.DataFrame(columns=["month_end", "n_currencies", "low_carry", "high_carry"])
    rows = []
    for dt, sub in panel.dropna(subset=["carry_signal_1m"]).groupby("month_end"):
        sub = sub.sort_values("carry_signal_1m")
        rows.append({
            "month_end": dt,
            "n_currencies": len(sub),
            "low_carry": ", ".join(sub.head(n)["currency"]),
            "high_carry": ", ".join(sub.tail(n).sort_values("carry_signal_1m", ascending=False)["currency"]),
        })
    return pd.DataFrame(rows)

rankings_by_month = top_bottom_by_month(monthly_panel, n=3)

display(Markdown("### First available months"))
display(rankings_by_month.head(12))

display(Markdown("### Most recent months"))
display(rankings_by_month.tail(12))


### First available months

,month_end,n_currencies,low_carry,high_carry
0,2007-01-31,9,"JPY, CHF, SEK","NZD, AUD, GBP"
1,2007-02-28,9,"JPY, CHF, SEK","NZD, AUD, GBP"
2,2007-03-31,9,"JPY, CHF, SEK","NZD, AUD, GBP"
3,2007-04-30,9,"JPY, CHF, SEK","NZD, AUD, GBP"
4,2007-05-31,9,"JPY, CHF, SEK","NZD, AUD, GBP"
5,2007-06-30,9,"JPY, CHF, SEK","NZD, AUD, GBP"
6,2007-07-31,9,"JPY, CHF, SEK","NZD, AUD, GBP"
7,2007-08-31,9,"JPY, CHF, SEK","NZD, AUD, GBP"
8,2007-09-30,9,"JPY, CHF, SEK","NZD, AUD, GBP"
9,2007-10-31,9,"JPY, CHF, EUR","NZD, AUD, GBP"


### Most recent months

,month_end,n_currencies,low_carry,high_carry
222,2025-07-31,9,"CHF, JPY, EUR","NOK, GBP, AUD"
223,2025-08-31,9,"CHF, JPY, EUR","NOK, GBP, AUD"
224,2025-09-30,9,"CHF, JPY, SEK","GBP, NOK, AUD"
225,2025-10-31,9,"CHF, JPY, SEK","NOK, GBP, AUD"
226,2025-11-30,9,"CHF, JPY, SEK","NOK, GBP, AUD"
227,2025-12-31,9,"CHF, JPY, SEK","GBP, NOK, AUD"
228,2026-01-31,9,"CHF, JPY, SEK","NOK, AUD, GBP"
229,2026-02-28,9,"CHF, JPY, SEK","NOK, AUD, GBP"
230,2026-03-31,9,"CHF, JPY, SEK","AUD, NOK, GBP"
231,2026-04-30,9,"CHF, JPY, SEK","AUD, NOK, GBP"


## Currency-Level Sanity Checks

Average carry and realized returns should not be interpreted as final performance. They are used here to catch sign errors, quote-direction mistakes, stale data, and missing observations before strategy construction.


In [10]:
if monthly_panel.empty:
    currency_summary = pd.DataFrame(columns=[
        "currency", "months", "first_month", "last_month", "avg_carry_signal_1m",
        "avg_realized_1m_forward_excess_return", "missing_spot_months", "missing_forward_months",
        "missing_realized_return_months",
    ])
else:
    expected_months = pd.date_range(monthly_panel["month_end"].min(), monthly_panel["month_end"].max(), freq="ME")
    rows = []
    for ccy in G10_BASELINE:
        sub = monthly_panel[monthly_panel["currency"] == ccy].copy()
        rows.append({
            "currency": ccy,
            "months": len(sub),
            "first_month": sub["month_end"].min().date().isoformat() if len(sub) else "",
            "last_month": sub["month_end"].max().date().isoformat() if len(sub) else "",
            "avg_carry_signal_1m": sub["carry_signal_1m"].mean() if len(sub) else np.nan,
            "avg_realized_1m_forward_excess_return": sub["realized_1m_forward_excess_return"].mean() if len(sub) else np.nan,
            "missing_spot_months": int(sub["spot_usd_per_ccy"].isna().sum()) if len(sub) else len(expected_months),
            "missing_forward_months": int(sub["forward_1m_usd_per_ccy"].isna().sum()) if len(sub) else len(expected_months),
            "missing_realized_return_months": int(sub["realized_1m_forward_excess_return"].isna().sum()) if len(sub) else len(expected_months),
        })
    currency_summary = pd.DataFrame(rows)

display(currency_summary)

display(Markdown("### Average carry by currency"))
display(currency_summary[["currency", "avg_carry_signal_1m"]].sort_values("avg_carry_signal_1m", ascending=False))

display(Markdown("### Average realized forward excess return by currency"))
display(currency_summary[["currency", "avg_realized_1m_forward_excess_return"]].sort_values("avg_realized_1m_forward_excess_return", ascending=False))


,currency,months,first_month,last_month,avg_carry_signal_1m,avg_realized_1m_forward_excess_return,missing_spot_months,missing_forward_months,missing_realized_return_months
0,AUD,234,2007-01-31,2026-06-30,0.001067,0.001213,0,0,1
1,CAD,234,2007-01-31,2026-06-30,-0.000121,-0.000594,0,0,1
2,CHF,234,2007-01-31,2026-06-30,-0.001672,0.000578,0,0,1
3,EUR,234,2007-01-31,2026-06-30,-0.000872,-0.001082,0,0,1
4,GBP,234,2007-01-31,2026-06-30,-0.000119,-0.001492,0,0,1
5,JPY,234,2007-01-31,2026-06-30,-0.001650,-0.002535,0,0,1
6,NOK,234,2007-01-31,2026-06-30,0.000292,-0.001088,0,0,1
7,NZD,234,2007-01-31,2026-06-30,0.001192,0.001063,0,0,1
8,SEK,234,2007-01-31,2026-06-30,-0.000649,-0.001550,0,0,1


### Average carry by currency

,currency,avg_carry_signal_1m
7,NZD,0.001192
0,AUD,0.001067
6,NOK,0.000292
4,GBP,-0.000119
1,CAD,-0.000121
8,SEK,-0.000649
3,EUR,-0.000872
5,JPY,-0.001650
2,CHF,-0.001672


### Average realized forward excess return by currency

,currency,avg_realized_1m_forward_excess_return
0,AUD,0.001213
7,NZD,0.001063
2,CHF,0.000578
1,CAD,-0.000594
3,EUR,-0.001082
6,NOK,-0.001088
4,GBP,-0.001492
8,SEK,-0.001550
5,JPY,-0.002535


## Save Intermediate Panel

The saved file is the clean data input for later strategy notebooks. It contains month-end spot, 1M forward, carry signal, and realized one-month forward excess return in a consistent USD-per-foreign-currency convention.


In [11]:
output_path = PROCESSED_DIR / "g10_1m_carry_panel.parquet"
fallback_csv_path = PROCESSED_DIR / "g10_1m_carry_panel.csv"

save_status = {"parquet_path": str(output_path.relative_to(ROOT)), "csv_fallback_path": "", "rows_saved": len(monthly_panel), "saved_as": ""}
try:
    monthly_panel.to_parquet(output_path, index=False)
    save_status["saved_as"] = "parquet"
except Exception as exc:
    warnings.warn(f"Could not save parquet file ({repr(exc)}). Saving CSV fallback instead.")
    monthly_panel.to_csv(fallback_csv_path, index=False)
    save_status["csv_fallback_path"] = str(fallback_csv_path.relative_to(ROOT))
    save_status["saved_as"] = "csv"

display(pd.DataFrame([save_status]))


,parquet_path,csv_fallback_path,rows_saved,saved_as
0,theo/data/processed/g10_1m_carry_panel.parquet,,2106,parquet


## Construction Notes and Next Checks

This notebook has built the core G10 1M carry panel only. The next notebook can use this panel to form portfolios, apply transaction costs, compare high-carry versus low-carry baskets, or add risk and option signals.

Before using the panel in a final backtest, manually review any ambiguous forward diagnostics, confirm the forward-points convention if Bloomberg settings change, and decide whether DKK/HKD should be added as a separate optional sample. Bid/ask data are available in the raw file, but transaction-cost-adjusted returns should be handled in a dedicated strategy notebook rather than mixed into this construction step.
